# 03 - Rolling-window training

Step 3 of `THESIS_GUIDE.md`. The six models are trained on the FinBench rolling protocol (4 years of training, 1 of validation, 1 of test; test years 2020-2024) in three universes (DJI, NASDAQ-100, EURO STOXX 50), with T = 20 and L = 5. Each model is trained **exactly as FinBench trains it** and keeps its weights with FinBench's own rule (decided 2026-10-06, `xaifin.training.trainer.SELECTION_RULES`); the data-quality filter is on.

This notebook reads the results store (`xaifin.training.rolling.collect_results`) and answers three questions:

1. **How well do the models predict?** IC, RankIC, ICIR and MSE per model, universe and test year.
2. **Are the numbers in FinBench's range?** FinBench publishes only MSE and MAE for these models (`results/analysis/finbench_table2.csv`).
3. **How weak is the signal?** The starting point of the thesis: if the predictions barely rank the stocks, explanations and portfolios built on them can be unstable.

Metrics are computed on the test year with the labels the model is scored on (the daily z-score of the forward return). **IC** and **RankIC** are the daily cross-sectional Pearson and Spearman correlations between predictions and labels, averaged over the test days; **ICIR** and **RankICIR** divide that mean by the standard deviation of the daily values. FactorVAE's label is a 20-day return (issue 3 in `docs/model_notes.md`), and FinFormer's loss ignores scale, so its MSE is not comparable (notebook 02).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from xaifin import viz
from xaifin.config import CORE_UNIVERSES, RESULTS_ROOT, TEST_YEARS, UNIVERSE_NAMES
from xaifin.training.rolling import collect_results

SEQ_LEN, PRED_LEN = 20, 5
MODELS = viz.MODEL_ORDER
all_runs = collect_results()
runs = all_runs[(all_runs["seq_len"] == SEQ_LEN) & (all_runs["pred_len"] == PRED_LEN) & all_runs["universe"].isin(CORE_UNIVERSES)]
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"{len(runs)} finished runs (T={SEQ_LEN}, L={PRED_LEN}), seeds {sorted(runs['seed'].unique())}")

252 finished runs (T=20, L=5), seeds [np.int64(0), np.int64(5), np.int64(42)]


## 1. The grid

One run per model, universe, test year and seed. EURO STOXX 50 has no 2020 run: its membership history starts on 2020-09-30, so no stock is a member at the start of 2020 (notebook 01). Training time is on one RTX 5090; FactorVAE is the slow one (FinBench's 96 attention layers run one after another for every day).

In [2]:
expected = {u: [y for y in TEST_YEARS if not (u == "sx5e" and y == 2020)] for u in CORE_UNIVERSES}
coverage = (runs.groupby(["model", "universe"]).size().unstack("universe").reindex(index=MODELS, columns=CORE_UNIVERSES)
            .fillna(0).astype(int))
coverage.loc["expected per model"] = [len(expected[u]) * runs["seed"].nunique() for u in CORE_UNIVERSES]
display(coverage)
timing = runs.groupby("model").agg(runs=("minutes", "size"), minutes_mean=("minutes", "mean"), minutes_total=("minutes", "sum"),
                                   epochs_run=("epochs_run", "mean"), kept_epoch=("kept_epoch", "mean")).reindex(MODELS)
timing

universe,dji,nasdaq100,sx5e
model,,,
MASTER,15,15,12
MATCC,15,15,12
FactorVAE,15,15,12
HIST,15,15,12
DiscoverPLF,15,15,12
FinFormer,15,15,12
expected per model,15,15,12


,runs,minutes_mean,minutes_total,epochs_run,kept_epoch
model,,,,,
MASTER,42,2.0000,84.0000,11.1905,10.1905
MATCC,42,15.9000,667.8000,70.0000,69.0000
FactorVAE,42,67.1595,2820.7000,30.0000,0.9048
HIST,42,11.0643,464.7000,55.7143,24.7143
DiscoverPLF,42,24.3762,1023.8000,48.8095,30.9524
FinFormer,42,5.3952,226.6000,17.1190,6.1190


## 2. Test performance per model, universe and year

Mean over the three seeds. Blue cells: the predictions rank the stocks in the right direction on average over the test year; red: the wrong direction.

In [3]:
cells = runs.groupby(["universe", "model", "test_year"])[["IC", "RankIC", "ICIR", "RankICIR", "MSE", "MAE"]].mean().reset_index()
for universe in CORE_UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if grid.empty:
        continue
    viz.metric_heatmap(grid, "RankIC", f"{UNIVERSE_NAMES[universe]}: test RankIC by model and test year").show()
    display(grid.pivot(index="model", columns="test_year", values="RankIC").reindex([m for m in MODELS if m in set(grid["model"])]))

test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0299,0.0194,-0.0155,0.0269,0.0146
MATCC,0.0266,0.0109,0.0237,0.0260,-0.0085
FactorVAE,0.0428,0.0868,-0.0590,0.0469,-0.0432
HIST,0.0383,0.0279,-0.0024,0.0304,0.0558
DiscoverPLF,0.0200,0.0127,0.0073,0.0380,0.0095
FinFormer,0.0333,0.0187,0.0189,0.0059,0.0414


test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0272,0.0052,-0.0034,0.0154,0.0058
MATCC,0.0388,0.0146,-0.0137,0.0182,0.0133
FactorVAE,0.0026,-0.0732,-0.0774,0.0185,-0.0362
HIST,0.0256,0.0036,-0.0073,0.0014,0.0080
DiscoverPLF,0.0242,-0.0061,-0.0173,0.0095,0.0127
FinFormer,0.0473,0.0049,-0.0267,-0.0026,-0.0111


test_year,2021,2022,2023,2024
model,,,,
MASTER,0.0062,-0.0184,-0.0291,-0.0195
MATCC,-0.0110,-0.0217,0.0119,-0.0085
FactorVAE,0.0477,0.0053,-0.0345,-0.0555
HIST,-0.0193,0.0222,0.0040,-0.0272
DiscoverPLF,0.0041,-0.0021,0.0066,-0.0419
FinFormer,0.0200,-0.0102,0.0034,-0.0125


In [4]:
for universe in CORE_UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if not grid.empty:
        viz.metric_by_year_chart(grid, "IC", f"{UNIVERSE_NAMES[universe]}: test IC per year").show()

## 3. Averages over the test years, and FinBench's numbers

Per model and universe: the mean over test years (and seeds) of each metric, and its standard deviation across years. FinBench's Table 2 reports MSE and MAE averaged over its rolling test years and seeds; its FactorVAE numbers come from the leaky readout (issue 1), and its MASTER, MATCC and FinFormer runs were not seeded (issue 8).

In [5]:
finbench = pd.read_csv(RESULTS_ROOT / "analysis" / "finbench_table2.csv").query("pred_len == @PRED_LEN")
summary = (runs.groupby(["universe", "model"])
           .agg(years=("test_year", "nunique"), IC=("IC", "mean"), IC_std=("IC", "std"), RankIC=("RankIC", "mean"),
                RankIC_std=("RankIC", "std"), ICIR=("ICIR", "mean"), RankICIR=("RankICIR", "mean"),
                MSE=("MSE", "mean"), MAE=("MAE", "mean"))
           .reset_index()
           .merge(finbench.rename(columns={"MSE": "FinBench MSE", "MAE": "FinBench MAE"})[["universe", "model", "FinBench MSE", "FinBench MAE"]],
                  on=["universe", "model"], how="left"))
summary["model"] = pd.Categorical(summary["model"], MODELS)
summary = summary.sort_values(["universe", "model"]).reset_index(drop=True)
summary

,universe,model,years,IC,IC_std,RankIC,RankIC_std,ICIR,RankICIR,MSE,MAE,FinBench MSE,FinBench MAE
0,dji,MASTER,5,0.0160,0.0275,0.0150,0.0243,0.0715,0.0673,0.9801,0.7464,1.2490,0.8590
1,dji,MATCC,5,0.0136,0.0229,0.0157,0.0183,0.0607,0.0724,1.4913,0.9417,1.1300,0.8180
2,dji,FactorVAE,5,0.0187,0.0648,0.0149,0.0596,0.1072,0.0784,0.8884,0.7247,0.9710,0.7580
3,dji,HIST,5,0.0288,0.0212,0.0300,0.0212,0.1229,0.1343,0.9851,0.7482,0.9970,0.7470
4,dji,DiscoverPLF,5,0.0154,0.0191,0.0175,0.0169,0.0696,0.0805,1.1870,0.8274,1.2110,0.8300
5,dji,FinFormer,5,0.0190,0.0178,0.0237,0.0198,0.0823,0.1044,2.4674,1.2304,1.6570,0.9820
6,nasdaq100,MASTER,5,0.0078,0.0121,0.0100,0.0137,0.0504,0.0661,1.0149,0.7177,1.0170,0.7200
7,nasdaq100,MATCC,5,0.0091,0.0164,0.0143,0.0195,0.0606,0.0994,1.7312,0.9889,1.7130,0.9810
8,nasdaq100,FactorVAE,5,-0.0338,0.0408,-0.0331,0.0416,-0.1921,-0.1794,0.9032,0.7183,0.9180,0.7250
9,nasdaq100,HIST,5,0.0030,0.0155,0.0063,0.0137,0.0113,0.0317,1.0345,0.7230,1.0300,0.7210


## 4. How weak is the signal?

Three views of the same question. (a) The share of runs whose test RankIC is above 0: a model with no skill lands there half of the time. (b) The t-statistic of the mean daily RankIC. The labels of consecutive days overlap (each is a 5-day return), so the daily values are correlated and RankICIR × √(test days) would overstate it; RankICIR × √(test days / L) counts only non-overlapping periods. About 2 or more means the year's mean is unlikely to be noise. (c) The best test RankIC per universe and year: the ceiling of what any of the six reaches.

In [6]:
days = 252  # test days per year
weak = runs.assign(t_stat=runs["RankICIR"] * np.sqrt(days / PRED_LEN))
share = (weak.groupby("model").agg(runs=("RankIC", "size"), positive_RankIC=("RankIC", lambda s: (s > 0).mean()),
                                   mean_RankIC=("RankIC", "mean"), significant=("t_stat", lambda s: (s.abs() >= 2).mean()))
         .reindex(MODELS))
display(share.style.format({"positive_RankIC": "{:.0%}", "significant": "{:.0%}", "mean_RankIC": "{:+.4f}"}))
best = weak.loc[weak.groupby(["universe", "test_year"])["RankIC"].idxmax(), ["universe", "test_year", "model", "RankIC", "t_stat"]]
best

,runs,positive_RankIC,mean_RankIC,significant
model,,,,
MASTER,42,55%,+0.0046,0%
MATCC,42,67%,+0.0086,2%
FactorVAE,42,40%,-0.0092,33%
HIST,42,67%,+0.0115,2%
DiscoverPLF,42,64%,+0.0055,2%
FinFormer,42,62%,+0.0093,2%


,universe,test_year,model,RankIC,t_stat
173,dji,2020,MASTER,0.0529,1.5343
48,dji,2021,FactorVAE,0.0963,3.2858
212,dji,2022,MATCC,0.0408,1.3533
50,dji,2023,FactorVAE,0.0711,2.8780
98,dji,2024,FinFormer,0.0724,2.6545
99,nasdaq100,2020,FinFormer,0.0528,1.8942
231,nasdaq100,2021,MATCC,0.0277,1.3263
190,nasdaq100,2022,MASTER,0.0102,0.3001
196,nasdaq100,2023,MASTER,0.0274,1.2987
234,nasdaq100,2024,MATCC,0.0195,1.0717


## 5. Seeds: same model, different training luck

Each cell (model, universe, test year) is trained with three seeds (0, 5, 42), which change only the random parts of training: the initial weights, the order of the training days, dropout. If the seeds of one model disagree as much as different models do, differences between models cannot be told apart from training luck. Per universe, averaged over test years: the spread across seeds within a model (standard deviation of RankIC, averaged over models) and the spread across models (standard deviation of the models' seed-averaged RankIC), and the share of cells where the seeds do not even agree on the sign of RankIC.

In [7]:
by_cell = runs.groupby(["universe", "test_year", "model"])["RankIC"]
cell = by_cell.agg(seed_mean="mean", seed_std="std", seeds="size",
                   sign_flip=lambda s: bool((s > 0).any() and (s < 0).any())).reset_index()
spread = (cell.groupby(["universe", "test_year"])
          .agg(within_model_seed_std=("seed_std", "mean"), across_model_std=("seed_mean", "std"),
               sign_disagreement=("sign_flip", "mean"))
          .groupby("universe").mean())
display(spread.style.format({"within_model_seed_std": "{:.4f}", "across_model_std": "{:.4f}", "sign_disagreement": "{:.0%}"}))
per_model = cell.groupby("model").agg(seed_std=("seed_std", "mean"), sign_disagreement=("sign_flip", "mean")).reindex(MODELS)
per_model.style.format({"seed_std": "{:.4f}", "sign_disagreement": "{:.0%}"})

,within_model_seed_std,across_model_std,sign_disagreement
universe,,,
dji,0.0143,0.0233,33%
nasdaq100,0.0096,0.0206,40%
sx5e,0.0118,0.0196,46%


,seed_std,sign_disagreement
model,,
MASTER,0.0141,43%
MATCC,0.0120,36%
FactorVAE,0.0141,14%
HIST,0.0085,43%
DiscoverPLF,0.0111,50%
FinFormer,0.0114,50%


## 6. The selection rules

Each model keeps its weights with FinBench's rule. If the validation score that picks the weights carried information about the test year, the kept epoch's validation IC and the test IC would be related; with a weak signal they need not be. The table shows, per model, the correlation across runs between the validation IC of the kept epoch and the test IC, and how many epochs the rule ran and kept.

In [8]:
selection = (runs.groupby("model")
             .apply(lambda d: pd.Series({"runs": len(d), "valid_IC_mean": d["valid_IC"].mean(), "test_IC_mean": d["IC"].mean(),
                                         "corr(valid IC, test IC)": d["valid_IC"].corr(d["IC"]) if len(d) > 2 else np.nan,
                                         "epochs_run": d["epochs_run"].mean(), "kept_epoch": d["kept_epoch"].mean()}),
                    include_groups=False)
             .reindex(MODELS))
selection

,runs,valid_IC_mean,test_IC_mean,"corr(valid IC, test IC)",epochs_run,kept_epoch
model,,,,,,
MASTER,42.0000,0.0061,0.0051,-0.1581,11.1905,10.1905
MATCC,42.0000,0.0126,0.0066,-0.0012,70.0000,69.0000
FactorVAE,42.0000,0.0145,-0.0084,-0.1906,30.0000,0.9048
HIST,42.0000,0.0369,0.0102,0.1068,55.7143,24.7143
DiscoverPLF,42.0000,0.0231,0.0038,0.0416,48.8095,30.9524
FinFormer,42.0000,0.0265,0.0076,0.1543,17.1190,6.1190


## 7. Saved summary

`results/analysis/metrics_summary.csv`: one row per run (cell, test metrics, kept epoch, epochs run, minutes, validation IC of the kept epoch). Later steps and the thesis tables read it.

In [9]:
out = RESULTS_ROOT / "analysis" / "metrics_summary.csv"
all_runs.drop(columns="path").sort_values(["model", "universe", "seq_len", "pred_len", "test_year", "seed"]).to_csv(out, index=False)
print(f"{len(all_runs)} runs -> {out}")

252 runs -> D:\CodeProjects\master-thesis\code\results\analysis\metrics_summary.csv


## Observations

The complete grid: 6 models x 3 universes x test years 2020-2024 (EURO STOXX 50 from 2021) x seeds 0, 5, 42, T = 20, L = 5, data-quality filter on.

1. **252 runs, no failures**, about 88 hours of GPU time on one RTX 5090. FactorVAE alone took 47 hours (67 minutes per run, FinBench's sequential attention layers); MASTER 2 minutes per run, FinFormer 5, HIST 11, MATCC 16, DiscoverPLF 24.
2. **The signal is weak everywhere.** Averaged over its 42 runs, each model's test RankIC is between -0.009 (FactorVAE) and +0.012 (HIST). RankIC is above 0 in 40% (FactorVAE) to 67% (MATCC, HIST) of the runs, where a model with no skill would land at about 50%. Counting only non-overlapping weeks, a run's mean RankIC is clearly different from 0 (|t| >= 2) in at most 1 of 42 runs for every model except FactorVAE (14 of 42, in both directions).
3. **It depends on the universe more than on the model.** On DJI every model's average RankIC is positive (0.015 to 0.030); on NASDAQ-100 it is between -0.033 (FactorVAE) and +0.014; on EURO STOXX 50 all six are at or below 0 (-0.015 to +0.000).
4. **No model wins consistently.** The best model of a universe and year changes from cell to cell (section 4, last table): five of the six models top at least one (all but DiscoverPLF), and the best run of a cell reaches a RankIC between 0.005 and 0.096.
5. **Training luck is half the size of the model differences.** Within a universe and year, the RankIC of one model's three seeds varies by 0.010-0.014 (standard deviation, averaged over models), against 0.020-0.023 across the six models; in 33-46% of the cells the three seeds of a model do not even agree on the sign of RankIC (section 5). FactorVAE's seeds agree most (sign disagreement 14%), because its rule keeps an almost untrained model.
6. **MSE is in FinBench's range** for MASTER (NASDAQ-100, EURO STOXX 50), FactorVAE, HIST and DiscoverPLF, within about 0.1 of FinBench's Table 2; MATCC's and FinFormer's are higher (MATCC 1.49-1.73, FinFormer 2.4-2.6; FinFormer's loss ignores scale), and MASTER's on DJI is lower (0.98 vs 1.25). FinBench's numbers average its own runs, partly unseeded and, for FactorVAE, scored differently, so only the range is comparable.
7. **The selection rules choose almost blindly.** Across runs, the validation IC of the kept epoch and the test IC are barely related (correlation -0.19 to 0.15 by model), and validation scores are optimistic (0.006-0.037 against -0.008 to 0.010 on the test year). FactorVAE's rule (lowest validation loss) keeps epoch 0 or 1 of 30 in almost every run.

Starting point for the thesis: the six architectures extract a small, unstable signal whose sign changes with the period, the universe and even the seed. Whether their explanations agree, and whether disagreement comes from the data or the models, is the question of Steps 5-7.